In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time
from numba import njit

# ================== 1. 物理内核：增强型演化引擎 ==================
@njit(fastmath=True, nogil=True)
def survey_kernel_v6(u_c, k_static, target_n, steps, n_bins):
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    # 增加热启动步数，确保概率流进入稳态
    warmup = 1000000 
    
    for i in range(steps + warmup):
        # 核心：1/ln^2 动力学引导
        ln_term = np.log(i + target_n + 100)
        mu_eff = u_c + k_static / (ln_term**2)
        
        x = 1 - mu_eff * x * x
        
        # 边界数值保护
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
        
        if i > warmup:
            current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
            if 0 <= current_bin < n_bins:
                if i == warmup + 1: last_bin = current_bin
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
                
    return counts

# ================== 2. 测绘 Worker ==================
def survey_worker_v6(task):
    target_n, k_val = task
    U_C = 1.543689012
    N_BINS = 10000    # 稍微提升分辨率
    STEPS = 5000000   # 提升到 500 万步，压低随机噪声
    
    SAVE_DIR = "riemann_10k_survey_high_density"
    
    try:
        t0 = time.time()
        counts = survey_kernel_v6(U_C, k_val, target_n, STEPS, N_BINS)
        
        P = sp.csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        # 提取前 120 个特征值
        vals, _ = eigs(P, k=120, which='LM', tol=1e-5)
        phases = np.sort(np.angle(vals[np.abs(vals) > 0.4]))
        
        filename = os.path.join(SAVE_DIR, f"n_{int(target_n)}_k_{k_val:.4f}.npy")
        np.save(filename, phases)
        
        return f"OK | n={target_n:5} | k={k_val:.4f} | {time.time()-t0:.1f}s"
    except Exception as e:
        return f"FAIL | n={target_n:5} | {str(e)}"

# ================== 3. 256 核全量分发 ==================
if __name__ == "__main__":
    SAVE_DIR = "riemann_10k_survey_high_density"
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)

    # --- 🌟 饱和攻击策略 🌟 ---
    # 8 个能级站位 (跨越 100 到 10000)
    # 每个站位部署 32 个探针 (覆盖 k=4.6 到 8.6)
    # 总任务数 = 8 * 32 = 256 (完美填满您的 256 核)
    n_stations = np.geomspace(100, 10000, 8) # 使用几何间距，更符合对数演化
    k_probes = np.linspace(4.6, 8.6, 32) 
    
    tasks = []
    for n in n_stations:
        for k in k_probes:
            tasks.append((n, k))

    print(f"="*60)
    print(f"🚀 启动 V6 高密度测绘：针对 beta 斜率修复")
    print(f">>> 核心数: 256 | 探针密度: 32/station | 步数: 5M")
    print(f"="*60)
    
    start_t = time.time()
    # 480G 内存建议维持 128 并发，给 ARPACK 特征值求解留出计算带宽
    with mp.Pool(processes=128) as pool:
        results = pool.map(survey_worker_v6, tasks)
    
    print(f"\n✅ 饱和测绘完成！总耗时: {(time.time()-start_t)/60:.2f} 分钟")

🚀 启动 V6 高密度测绘：针对 beta 斜率修复
>>> 核心数: 256 | 探针密度: 32/station | 步数: 5M

✅ 饱和测绘完成！总耗时: 9.71 分钟
